# Tutorial: `qcodes.measure_v2` (experimental)

This notebook walks through the experimental parallel measurement API in
QCoDeS, `qcodes.measure_v2`. It is a complement to (not a replacement for)
the existing `Measurement` / `dond` workflow.

**What's new in `measure_v2`:**

- **Plans are data, not for-loops.** A "plan" is a Python generator that
  yields typed messages (`Set`, `Read`, `Sleep`, `Emit`, …). A
  `MeasurementEngine` drives the plan on a dedicated worker thread.
- **Non-blocking submission.** `engine.submit(plan)` returns immediately
  with a `RunHandle`. The notebook kernel stays responsive while the scan
  runs.
- **Cancellation with guaranteed cleanup.** `handle.cancel()` runs the
  plan's `try/finally` blocks, so user-defined cleanup (e.g., ramping a
  gate back to 0) always executes — even on Ctrl-C.
- **Pluggable sinks.** A "sink" is just a callable that consumes events
  (`RunStarted`, `RowEmitted`, `RunStopped`). The default sink writes to
  the existing QCoDeS SQLite dataset; custom sinks can drive live plots,
  network dashboards, ML feedback loops, etc.

**Stability warning:** the `measure_v2` package is experimental. The API
may change in incompatible ways between releases. For production use,
stick with `Measurement` / `dond`.

The architecture document lives at `src/qcodes/measure_v2/DESIGN.md`.

## Setup

We'll use a temporary database file and a couple of software-only
parameters so the notebook is fully self-contained — no instruments
required.

In [1]:
from __future__ import annotations

import os
import tempfile
import time
from typing import TYPE_CHECKING

import numpy as np

import qcodes as qc
from qcodes import measure_v2 as mv2
from qcodes.dataset.dond.sweeps import LinSweep
from qcodes.dataset.sqlite.database import initialise_or_create_database_at

if TYPE_CHECKING:
    from collections.abc import Generator
    from typing import Any

    from qcodes.measure_v2 import Msg
    from qcodes.parameters import ParameterBase

# Temporary database for this notebook
tmpdir: str = tempfile.mkdtemp(prefix="measure_v2_tutorial_")
db_path: str = os.path.join(tmpdir, "tutorial.db")
qc.config["core"]["db_location"] = db_path
initialise_or_create_database_at(db_path)
print(f"Database: {db_path}")

Database: C:\Users\jenielse\AppData\Local\Temp\measure_v2_tutorial_7j91qifa\tutorial.db


In [2]:
# Software-only parameters. ``i`` is a function of ``g`` and ``h``,
# mimicking an instrument whose measurement depends on the gate voltages.
g: ParameterBase = qc.Parameter("g", initial_value=0.0, set_cmd=None, get_cmd=None)
h: ParameterBase = qc.Parameter("h", initial_value=0.0, set_cmd=None, get_cmd=None)
i: ParameterBase = qc.Parameter(
    "i",
    get_cmd=lambda: np.sin(g.cache.get()) * np.cos(h.cache.get()),
)

## 1. The simplest case: a blocking 1D scan

`qc.measure_v2.scan(...)` mirrors the role of `do1d` / `do2d` / `dond`.
By default it blocks until the run completes and returns the resulting
dataset.

In [3]:
ds = mv2.scan(
    LinSweep(g, 0.0, 2 * np.pi, 21),
    measure=[i],
    name="1d_blocking",
)
assert ds is not None  # tells the type checker this isn't None
print(type(ds).__name__)
print(ds.cache.data())

Starting experimental run with id: 1. 
DataSet
{'i': {'i': array([ 0.00000000e+00,  3.09016994e-01,  5.87785252e-01,  8.09016994e-01,
        9.51056516e-01,  1.00000000e+00,  9.51056516e-01,  8.09016994e-01,
        5.87785252e-01,  3.09016994e-01,  1.22464680e-16, -3.09016994e-01,
       -5.87785252e-01, -8.09016994e-01, -9.51056516e-01, -1.00000000e+00,
       -9.51056516e-01, -8.09016994e-01, -5.87785252e-01, -3.09016994e-01,
       -2.44929360e-16]), 'g': array([0.        , 0.31415927, 0.62831853, 0.9424778 , 1.25663706,
       1.57079633, 1.88495559, 2.19911486, 2.51327412, 2.82743339,
       3.14159265, 3.45575192, 3.76991118, 4.08407045, 4.39822972,
       4.71238898, 5.02654825, 5.34070751, 5.65486678, 5.96902604,
       6.28318531])}}


## 2. Non-blocking submission with `wait=False`

Pass `wait=False` to keep the kernel responsive while a long scan runs.
You get a `RunHandle` back instead of a dataset. The handle exposes
`wait()`, `cancel()`, `status`, and futures for the dataset and final
result.

In [4]:
handle: mv2.RunHandle = mv2.scan(
    LinSweep(g, 0.0, 2 * np.pi, 200, delay=0.01),
    measure=[i],
    name="1d_nonblocking",
    wait=False,
)
print("status while running:", handle.status)

# The notebook kernel can do other work here — e.g., peek partial data
# via the in-memory cache.
time.sleep(0.5)
partial_ds = handle.dataset.result(timeout=1.0)
assert partial_ds is not None
print(f"rows so far: {len(partial_ds.cache.data()['i']['g'])}")

# Then wait for it to complete.
result: mv2.RunResult = handle.wait(timeout=30.0)
print(f"final reason: {result.reason}, n_rows: {result.n_rows_emitted}")

status while running: running
Starting experimental run with id: 2. 


rows so far: 32


final reason: completed, n_rows: 200


## 3. Cancellation with safe cleanup

The architecture's central safety guarantee: when you cancel a run,
the plan's `try/finally` blocks always run. For built-in plan-builders
like `scan_1d` and `scan_nd`, the `finally` block ramps every swept
parameter back to `0.0` before reporting the run stopped.

This is verifiable: after a mid-flight cancel, the swept parameter's
cache shows `0.0`, not the value it was at when we cancelled.

In [5]:
# Start a long scan, drive g away from 0.
handle = mv2.scan(
    LinSweep(g, 0.0, 2.0, 1000, delay=0.01),
    measure=[i],
    name="cancel_safety_demo",
    wait=False,
)
time.sleep(0.05)

# Mid-flight: g is somewhere between 0 and 2.
print(f"g during scan: {g.cache.get():.3f}")

# Cancel. The plan's finally block runs and ramps g back to 0.
handle.cancel()
result = handle.wait(timeout=5.0)

print(f"reason: {result.reason}")
print(f"g after cancel: {g.cache.get():.3f}  (back to 0 — finally ran)")
print(f"partial dataset rows: {result.n_rows_emitted}")

Starting experimental run with id: 3. 
g during scan: 0.006
reason: cancelled
g after cancel: 0.000  (back to 0 — finally ran)
partial dataset rows: 3


## 4. N-dimensional scans with `scan_nd`

For multi-dimensional scans, just pass more sweeps. Outermost-first
ordering: the first sweep is the slowest axis.

In [6]:
ds = mv2.scan(
    LinSweep(g, 0.0, 2 * np.pi, 11),  # outer
    LinSweep(h, 0.0, 2 * np.pi, 11),  # inner
    measure=[i],
    name="2d_scan",
)
assert ds is not None
print(ds.description.interdeps)
print(f"shape: {len(ds.cache.data()['i']['g'])} rows")

Starting experimental run with id: 4. 
InterDependencies_(dependencies={ParamSpecBase('i', 'numeric', 'i', ''): (ParamSpecBase('g', 'numeric', 'g', ''), ParamSpecBase('h', 'numeric', 'h', ''))}, inferences={}, standalones=frozenset())
shape: 121 rows


## 5. Writing your own plan-builder

A plan is just a Python generator that yields message dataclasses. You
write the loop; the engine handles threading, dispatch, cancellation,
and event publication.

The contract:
- Yield `Set(param, value)`, `Sleep(seconds)`, `Read((param, ...))`, or
  `Emit()` messages.
- Use `result = yield Read((...))` to get the read values back into your
  plan — this is what enables adaptive scans.
- Put cleanup in a `try/finally` block. The engine guarantees the
  `finally` runs on cancel.

Example: a **bisection search** for the gate voltage where a signal
crosses a threshold. The plan decides each next setpoint based on the
previous reading. We use `cos(g)`, which crosses 0 at `g = π/2`.

In [7]:
def bisect_search(
    gate: ParameterBase,
    signal: ParameterBase,
    *,
    lo: float,
    hi: float,
    threshold: float,
    tolerance: float,
) -> Generator[Msg, Any, None]:
    """Plan-builder: binary-search for the threshold crossing."""
    try:
        while abs(hi - lo) > tolerance:
            mid = 0.5 * (hi + lo)
            yield mv2.Set(gate, mid)
            yield mv2.Sleep(0.001)
            r: dict[ParameterBase, Any] = yield mv2.Read((signal,))
            yield mv2.Emit()
            # cos is monotonically decreasing on [0, π]: shrink lo or hi
            # to keep the threshold crossing bracketed.
            if r[signal] < threshold:
                hi = mid
            else:
                lo = mid
    finally:
        yield mv2.Set(gate, 0.0)  # always return to safe value


# Engineer a signal that crosses 0 at g = π/2 (the canonical bisection target
# on [0, π] for cos).
target: ParameterBase = qc.Parameter("target", get_cmd=lambda: np.cos(g.cache.get()))

Plan-builders that yield their own messages need to be wrapped with
the `run(...)` decorator to add the `OpenRun`/`CloseRun` lifecycle, then
submitted to an engine:

In [8]:
plan = mv2.run(
    name="bisect_demo",
    setpoints=(g,),
    measured=(target,),
)(bisect_search(g, target, lo=0.0, hi=np.pi, threshold=0.0, tolerance=0.001))

engine: mv2.MeasurementEngine = mv2.default_engine()
handle = engine.submit(plan)
handle.wait(timeout=10.0)

ds = handle.dataset.result()
assert ds is not None
data = ds.cache.data()
final_g: float = float(list(data["target"]["g"])[-1])
print(f"crossing found near g = {final_g:.4f}  (expected: π/2 ≈ {np.pi / 2:.4f})")

Starting experimental run with id: 5. 


crossing found near g = 1.5716  (expected: π/2 ≈ 1.5708)


## 6. Custom sinks for live data

A "sink" is any callable that accepts an `Event`. Sinks run on the
engine's publisher thread; they're the natural place to attach live
plotting, network dashboards, ML feedback loops, etc.

The simplest sink is `MemorySink`, which just records every event into
a list — perfect for inspecting what the engine actually emits.

In [9]:
# Build an engine with both a SqliteSink AND a MemorySink.
sink: mv2.MemorySink = mv2.MemorySink()
engine = mv2.MeasurementEngine(
    sinks=[mv2.SqliteSink(experiment_name="custom_sinks"), sink],
)
try:
    plan = mv2.run(setpoints=(g,), measured=(i,))(
        mv2.scan_1d(LinSweep(g, 0.0, 1.0, 5), [i])
    )
    engine.submit(plan).wait(timeout=10.0)
finally:
    engine.shutdown(wait=True, timeout=5.0)

# Inspect the event stream.
print(f"total events: {len(sink.events)}")
print(f"  RunStarted: {len(sink.starts)}")
print(f"  RowEmitted: {len(sink.rows)}")
print(f"  RunStopped: {len(sink.stops)}")
print()
print("first emitted row's snapshot:")
print({p.name: v for p, v in sink.rows[0].snapshot.items()})

Starting experimental run with id: 6. 
total events: 7
  RunStarted: 1
  RowEmitted: 5
  RunStopped: 1

first emitted row's snapshot:
{'g': 0.0, 'i': np.float64(0.0)}


### Writing a custom sink

A sink is just a callable. Want a real-time progress printer? One
function:

In [10]:
def progress_sink(event: mv2.Event) -> None:
    """Print a status line for each event."""
    name: str = type(event).__name__
    if isinstance(event, mv2.RowEmitted):
        print(f"  row #{event.seq}")
    else:
        print(f"  {name}: {event}")


engine = mv2.MeasurementEngine(
    sinks=[mv2.SqliteSink(experiment_name="custom_sinks"), progress_sink],
)
try:
    plan = mv2.run(setpoints=(g,), measured=(i,))(
        mv2.scan_1d(LinSweep(g, 0.0, 1.0, 3), [i])
    )
    engine.submit(plan).wait(timeout=10.0)
finally:
    engine.shutdown(wait=True, timeout=5.0)

Starting experimental run with id: 7. 
  RunStarted: RunStarted(run_id=UUID('2c727ca7-931c-4f59-a437-dcd8db54b8c6'), name='', descriptor=Descriptor(setpoints=(<qcodes.parameters.parameter.Parameter: g at 2852958764400>,), measured=(<qcodes.parameters.parameter.Parameter: i at 2853534537664>,), shapes=None), exp=None, write_period=None, started_at=1778575446.3018694)
  row #0
  row #1
  row #2
  RunStopped: RunStopped(run_id=UUID('2c727ca7-931c-4f59-a437-dcd8db54b8c6'), reason='completed', error=None, started_at=1778575446.3018694, stopped_at=1778575446.3018694, cancel_latency=None, n_rows_emitted=3)


## 7. Unit-testing plan-builders without an engine

Because plans are pure generators, you can unit-test them without
instantiating an engine, sinks, or instruments. We can drive the plan
generator manually, synthesizing responses to `Read` messages and
recording every yielded message.

(The `qcodes.measure_v2.testing.drive_plan` helper does this for
trivial cases. Plans that need stateful response synthesis are easiest
to drive by hand, as shown here.)

In [11]:
# A fake parameter pair. We'll drive bisect_search against a signal
# whose sign flips at g = 1.0; bisection should converge near 1.0.
fake_g: ParameterBase = qc.Parameter("fake_g")
fake_signal: ParameterBase = qc.Parameter("fake_signal")

state: dict[str, float] = {"last_g": 0.0}


def custom_drive() -> list[Msg]:
    """Drive bisect_search to completion, synthesizing Read responses."""
    plan: Generator[Msg, Any, None] = bisect_search(
        fake_g,
        fake_signal,
        lo=0.0,
        hi=2.0,
        threshold=0.0,
        tolerance=0.01,
    )
    msg: Msg = next(plan)
    messages: list[Msg] = [msg]
    send_value: Any = None
    try:
        while True:
            if isinstance(msg, mv2.Set) and msg.param is fake_g:
                state["last_g"] = float(msg.value)
            if isinstance(msg, mv2.Read):
                send_value = {
                    fake_signal: 1.0 if state["last_g"] < 1.0 else -1.0,
                }
            else:
                send_value = None
            msg = plan.send(send_value)
            messages.append(msg)
    except StopIteration:
        pass
    return messages


messages: list[Msg] = custom_drive()
sets: list[mv2.Set] = [m for m in messages if isinstance(m, mv2.Set)]
sweep_sets: list[mv2.Set] = sets[:-1]  # last is the cleanup ramp-to-0
print(f"number of bisection iterations: {len(sweep_sets)}")
final_sweep_g: float = float(sweep_sets[-1].value)
print(f"plan converged near g = {final_sweep_g:.4f}  (expected: 1.0)")

number of bisection iterations: 8
plan converged near g = 0.9922  (expected: 1.0)


## 8. Things to watch out for

- **Experimental status.** API may change in incompatible ways.
- **One run at a time per engine** in v1 — concurrent `submit()` raises
  `RuntimeError`. Queuing is on the v1 backlog.
- **`Sleep` is cancellable** in ~100ms chunks; `Set`/`Read` are not
  interruptible mid-call. A scan with 60-second per-point delays takes
  up to 60s to cancel.
- **`dataset.cache.data()` is the safe way to read live data** from the
  notebook thread. `dataset.get_parameter_data()` uses the connection
  bound to the engine's publisher thread and will fail with a thread
  mismatch from the main thread.
- **Things NOT in this release**: `enter_actions`/`exit_actions`,
  inner-loop actions, `break_condition`, `additional_setpoints`,
  `flush_columns`, `live_plot=True` sugar, queueing, pause/resume,
  parallel reads via `underlying_instrument` grouping. They're all
  on the v1 P0/P1 backlog and don't disturb the architectural foundation.

## Further reading

- Architecture document: `src/qcodes/measure_v2/DESIGN.md`
- Decisions log: same document, §13
- Tests: `tests/measure_v2/` — particularly `test_acceptance.py` for
  end-to-end examples.